# Meridian Commerce Platform ? Interactive Multi-Cluster GKE & Observability Demo
This notebook runs in **Google Colab**, **JupyterLab on Windows**, or **macOS** without paid cloud resources. It demonstrates:
1. Local storefront and order-service request simulations with trace IDs.
2. Rolling six-hour multi-cluster telemetry for primary and secondary regions.
3. Local SQL analyses for error rates, restarts, latency, resources, and HTTP error traces.
4. A five-panel Grafana-style dashboard with four deterministic container error scenarios.

This notebook is a local simulation; it does not query BigQuery or fetch live Google Cloud logs. For BigQuery table queries and Grafana import, see `observability/bigquery_queries.sql` and `observability/grafana-dashboard.json`.


In [ ]:
# Cell 1: Verify Web App A (storefront-ui) and Web App B (order-service) Endpoints (Zero external pip dependencies!)
import json, uuid, time, random, math, sqlite3
from datetime import datetime, timedelta, timezone

CATALOG = [
    {'sku': 'MRD-101', 'name': 'Titanium Field Chronograph', 'price': 245.00, 'stock': 42},
    {'sku': 'MRD-204', 'name': 'All-Weather Expedition Pack 45L', 'price': 189.00, 'stock': 19},
    {'sku': 'MRD-309', 'name': 'Solar Telemetry Beacon', 'price': 320.00, 'stock': 11},
]

def storefront_ui_request(path):
    trace_id = uuid.uuid4().hex[:16]
    if path == '/healthz':
        return 200, {'status': 'healthy', 'service': 'storefront-ui', 'cluster': 'meridian-gke-primary', 'region': 'us-central1'}
    return 200, {'service': 'storefront-ui', 'cluster': 'meridian-gke-primary', 'items': CATALOG, 'trace_id': trace_id}

def order_service_request(payload):
    trace_id = uuid.uuid4().hex[:16]
    return 201, {'order_id': 'ORD-' + uuid.uuid4().hex[:6].upper(), 'sku': payload['sku'], 'status': 'CONFIRMED', 'trace_id': trace_id}

print('GET  storefront-ui /healthz     ->', storefront_ui_request('/healthz'))
print('GET  storefront-ui /api/catalog ->', storefront_ui_request('/api/catalog')[1]['items'][0])
print('POST order-service /api/orders  ->', order_service_request({'sku': 'MRD-101', 'quantity': 2}))

In [ ]:
# Cell 2: Generate rolling multi-cluster telemetry and run local SQL analyses
random.seed(42)
now = datetime.now(timezone.utc).replace(second=0, microsecond=0)
start_time = now - timedelta(hours=6)

clusters = [('meridian-gke-primary', 'us-central1', 0.62), ('meridian-gke-secondary', 'us-east1', 0.38)]
services = [
    ('storefront-ui', ['storefront-ui-7c8d9f-x92k1', 'storefront-ui-8b2a1c-q44z9']),
    ('order-service', ['order-service-5f4b2a-m19p4', 'order-service-9d3e7f-w81v2']),
]
logs = []
for step in range(72):
    bucket_ts = start_time + timedelta(minutes=step * 5)
    is_spike = 42 <= step <= 47
    for cluster_name, region, weight in clusters:
        for service_name, pods in services:
            event_count = max(1, int(random.randint(28, 42) * weight * (1.5 if is_spike else 1.0)))
            for _ in range(event_count):
                err_prob = 0.145 if (is_spike and cluster_name == 'meridian-gke-primary' and service_name == 'order-service') else (0.068 if is_spike and service_name == 'storefront-ui' else 0.012)
                is_err = random.random() < err_prob
                base_lat = random.lognormvariate(math.log(32.0 if service_name == 'storefront-ui' else 52.0), 0.4)
                latency_ms = round(min(base_lat * (2.5 if is_err or is_spike else 1.0), 950.0), 2)
                logs.append((
                    bucket_ts.strftime('%H:%M'), 'ERROR' if is_err else 'INFO', service_name,
                    cluster_name, 'meridian-prod', random.choice(pods),
                    (503 if service_name == 'storefront-ui' else 500) if is_err else (200 if service_name == 'storefront-ui' else 201),
                    latency_ms, round(random.gauss(78.0 if is_spike else 45.0, 6.0), 2),
                    round(random.gauss(310.0 if is_spike else 215.0, 14.0), 1),
                    'HTTP_REQUEST', uuid.uuid4().hex, 'SIMULATED_HTTP_ERROR' if is_err else 'OK', None
                ))

injected_errors = [
    ('meridian-gke-primary', 'us-central1', 'storefront-ui', 'storefront-ui-7c8d9f-x92k1', 503, 'CATALOG_REPLICA_TIMEOUT', 'a1010000000000000000000000000001'),
    ('meridian-gke-primary', 'us-central1', 'order-service', 'order-service-5f4b2a-m19p4', 500, 'INVENTORY_LOCK_TIMEOUT', 'a1010000000000000000000000000002'),
    ('meridian-gke-secondary', 'us-east1', 'storefront-ui', 'storefront-ui-8b2a1c-q44z9', 503, 'CATALOG_REPLICA_TIMEOUT', 'a1010000000000000000000000000003'),
    ('meridian-gke-secondary', 'us-east1', 'order-service', 'order-service-9d3e7f-w81v2', 500, 'INVENTORY_LOCK_TIMEOUT', 'a1010000000000000000000000000004'),
]
error_bucket = (now - timedelta(minutes=5)).strftime('%H:%M')
for cluster_name, region, service, pod, status, reason, trace_id in injected_errors:
    logs.append((error_bucket, 'ERROR', service, cluster_name, 'meridian-prod', pod, status, 850.0, 78.0, 360.0, 'HTTP_REQUEST', trace_id, reason, 'injected_container_error'))

conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE gke_logs (time_bucket TEXT, severity TEXT, service TEXT, cluster_name TEXT, namespace TEXT, pod_name TEXT, status_code INT, latency_ms REAL, cpu_pct REAL, mem_mb REAL, event_type TEXT, trace_id TEXT, reason TEXT, error_scenario TEXT)')
conn.executemany('INSERT INTO gke_logs VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)', logs)

print('--- Error rate summary by cluster and service ---')
for row in conn.execute("SELECT cluster_name, service, COUNT(*), SUM(CASE WHEN severity='ERROR' OR status_code>=500 THEN 1 ELSE 0 END), ROUND(AVG(CASE WHEN severity='ERROR' OR status_code>=500 THEN 100.0 ELSE 0.0 END), 2) FROM gke_logs WHERE error_scenario IS NULL GROUP BY 1, 2 ORDER BY 1, 2"):
    print(f'{row[0]:24s} | {row[1]:15s} | Total: {row[2]:5d} | Errors: {row[3]:3d} | Error Rate: {row[4]}%')

print('\n--- Injected HTTP errors and trace IDs ---')
for row in conn.execute("SELECT cluster_name, service, pod_name, status_code, reason, trace_id FROM gke_logs WHERE error_scenario='injected_container_error' ORDER BY cluster_name, service"):
    print(f'{row[0]} | {row[1]} ({row[2]}) | HTTP {row[3]} {row[4]} | trace_id={row[5]}')


In [ ]:
# Cell 3: Render the five-panel observability dashboard inline
import matplotlib.pyplot as plt
import numpy as np

buckets = sorted({row[0] for row in logs})
err_sf, err_or, p50, p95, p99, cpu_primary, cpu_secondary = [], [], [], [], [], [], []
for bucket in buckets:
    rows = [row for row in logs if row[0] == bucket and row[13] is None]
    sf = [row for row in rows if row[2] == 'storefront-ui']
    orders = [row for row in rows if row[2] == 'order-service']
    err_sf.append(sum(row[6] >= 500 for row in sf) * 100.0 / max(1, len(sf)))
    err_or.append(sum(row[6] >= 500 for row in orders) * 100.0 / max(1, len(orders)))
    latencies = [row[7] for row in rows]
    p50.append(np.percentile(latencies, 50)); p95.append(np.percentile(latencies, 95)); p99.append(np.percentile(latencies, 99))
    primary = [row[8] for row in rows if row[3] == 'meridian-gke-primary']
    secondary = [row[8] for row in rows if row[3] == 'meridian-gke-secondary']
    cpu_primary.append(np.mean(primary) if primary else 0)
    cpu_secondary.append(np.mean(secondary) if secondary else 0)

fig = plt.figure(figsize=(15, 12), facecolor='#111217')
grid = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.85], hspace=0.42, wspace=0.3, left=0.07, right=0.96, top=0.9, bottom=0.05)
fig.suptitle('Meridian Multi-Cluster GKE Observability ? Local Simulation', color='white', fontsize=15, fontweight='bold')
ax1, ax2, ax3, ax4 = (fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1]))
for ax in (ax1, ax2, ax3, ax4):
    ax.set_facecolor('#181b1f'); ax.tick_params(colors='#9aa0a6'); ax.grid(True, color='#26292e', linestyle='--')
x = np.arange(len(buckets))
ax1.plot(x, err_or, color='#f2495c', label='order-service 5xx %'); ax1.plot(x, err_sf, color='#ff9830', label='storefront-ui 5xx %')
ax1.set_title('1. Application HTTP 5xx Error Rate', color='white'); ax1.legend()
ax2.barh(['order-service primary', 'storefront-ui primary', 'order-service secondary', 'storefront-ui secondary', 'monitoring / ingress'], [4, 2, 1, 1, 3], color=['#f2495c', '#ff9830', '#73bf69', '#73bf69', '#ff9830'])
ax2.set_title('2. Pod Restart Counts', color='white')
ax3.plot(x, p99, color='#b877d9', label='p99 ms'); ax3.plot(x, p95, color='#5794f2', label='p95 ms'); ax3.plot(x, p50, color='#73bf69', label='p50 ms')
ax3.set_title('3. Request Latency Percentiles', color='white'); ax3.legend()
ax4.plot(x, cpu_primary, color='#5794f2', label='Primary CPU %'); ax4.plot(x, cpu_secondary, color='#73bf69', label='Secondary CPU %'); ax4.axhline(65, color='#f2495c', linestyle='--', label='HPA threshold')
ax4.set_title('4. Resource Utilization & Regional Failover', color='white'); ax4.legend()

error_ax = fig.add_subplot(grid[2, :]); error_ax.set_facecolor('#181b1f'); error_ax.axis('off')
error_ax.set_title('5. HTTP Error Scenarios & Trace IDs', loc='left', color='#f2495c', fontsize=10, fontweight='bold')
error_rows = [row for row in logs if row[13] == 'injected_container_error']
error_cells = [[row[3], row[2], row[5], str(row[6]), row[12], row[11]] for row in error_rows]
table = error_ax.table(cellText=error_cells, colLabels=['Cluster', 'Service', 'Pod', 'HTTP', 'Reason', 'Trace ID'], cellLoc='left', loc='center', colWidths=[0.2, 0.13, 0.23, 0.06, 0.18, 0.2], bbox=[0, 0, 1, 0.82])
table.auto_set_font_size(False); table.set_fontsize(7)
for (row_index, _), cell in table.get_celld().items():
    cell.set_edgecolor('#2c3235'); cell.set_text_props(color='#e8eaed', family='monospace' if row_index else 'sans-serif')
    cell.set_facecolor('#252a30' if row_index == 0 else ('#181b1f' if row_index % 2 else '#20242a'))
plt.show()
